# 08 | The economics model: two cohorts, channel CAC, break-even and gates

**Author: Chanakya**

The single source of truth for every economic figure in the recommendation. Two mutually exclusive cohorts are each measured against their own holdout. **Acquisition** buys new ATP pass buyers and is counted on an incremental basis. **Upgrade** moves existing ATP pass buyers to a season pass, and only upgrades above the control rate are credited, with the credit charged to every treated upgrader. The model lives in `models/atp_economics.py`; every input, with its evidence tag, is in `models/model_inputs.json`. Rights fees are excluded: this is an incremental campaign P&L, not a claim about total rights ROI.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='08_economics_model'
shared.ACTIVE_SOURCES=[]

Offline inputs: raw-v5-2026-09-14 | Author: Chanakya


In [2]:
sys.path.insert(0, str(ROOT/'models'))
import atp_economics as m
res = m.run()
rows=[]
for section,items in m.INPUTS.items():
 if section.startswith('_'):continue
 for key,x in items.items():
  if key.startswith('_'):continue
  rows.append(dict(section=section,input=key,value=json.dumps(x['value']) if isinstance(x['value'],(list,dict)) else x['value'],tag=x.get('tag',''),source=x.get('source',''),note=x.get('note','')))
inputs=pd.DataFrame(rows);display(table(inputs,'08_model_inputs'))
print('Evidence tags:',inputs.tag.value_counts().to_dict())

,section,input,value,tag,source,note
0,unit_economics,gst_rate,0.180,A,,GST treated as inside the listed price
1,unit_economics,gateway_fee,0.020,O,v3_razorpay_pricing,"2% of transaction, plus 18% GST on the fee"
2,unit_economics,gateway_fee_tax,0.180,O,v3_razorpay_pricing,
3,unit_economics,variable_cost_per_purchase,30,A,,"Delivery and other variable cost per purchase,..."
4,unit_economics,tournament_pass_price,89,C,,Midpoint of the brief's INR 79-99 tournament pass
...,...,...,...,...,...,...
54,experiments,alpha_two_sided,0.050,A,,
55,experiments,power,0.800,A,,
56,experiments,owned_pilot_baseline_conversion,0.010,A,,
57,experiments,owned_pilot_true_lift_pp,0.008,A,,


Evidence tags: {'A': 42, 'C': 7, 'O': 6, 'D': 2, 'O/A': 1, 'A/C': 1}


## 1. Unit economics
Contribution = price / 1.18 GST, less a 2% gateway fee plus GST on the fee, less INR 30 variable cost. The upgrade credit costs less than its INR 44.50 face value because GST and the gateway fee fall with the price.

In [3]:
u=pd.DataFrame([dict(item=k,value=v) for k,v in res['unit_economics'].items()]);display(table(u,'08_unit_economics'))

,item,value
0,pass_contribution,43.323
1,season_contribution,298.719
2,credit_cost_per_upgrade,36.662
3,credit_hurdle_rate_at_10pct_control,0.114
4,discount_10pct_volume_hurdle,0.111
5,purchases_per_buyer,1.625
6,contribution_per_buyer_year1,70.400
7,contribution_per_acquired_payer_24m,98.561
8,passes_to_repay_175,4.039
9,seasons_to_repay_175,0.586


## 2. Channel economics and the INR 3.04 Cr conditional envelope
Channels that clear the brief’s INR 150–200 target on attributed CAC keep their envelope. Paid media, at INR 333, is capped to a INR 25 lakh external-audience test. The freed money is not moved into owned messaging, because owned reach is finite and its incremental CAC rises with depth. It becomes a **performance reserve**, released only to a channel whose measured marginal incremental CAC is at or below INR 200. Test-ceiling channels have no public cost basis; their CAC is a purchasing rule, not an estimate.

In [4]:
ch=pd.DataFrame(res['channels']);display(table(ch,'08_channel_cac_and_net'))
env=pd.DataFrame([dict(item=k,value=v) for k,v in res['envelope'].items() if not isinstance(v,dict)]);display(table(env,'08_envelope_summary'))
gates=pd.DataFrame(res['gate_allocation']);display(table(gates,'08_gate_allocation'))
fig_,ax=plt.subplots(figsize=(10,4.2))
order=ch.sort_values('net_per_payer_24m_at_100')
ax.barh(order.label,order.net_per_payer_24m_at_100,color=[COLORS[3] if x< -1 else COLORS[1] for x in order.net_per_payer_24m_at_100],label='All attributed payers incremental')
ax.scatter(order.net_per_payer_24m_at_87_5,order.label,color='black',zorder=3,label='87.5% incremental')
ax.axvline(0,color='black',lw=.8);ax.set_xlabel('24-month contribution minus attributed CAC, INR per payer, before any upgrade')
ax.set_title('No acquisition channel pays back on pass purchases alone. Owned comes closest');ax.legend(loc='lower right',fontsize=8)
fig('08_channel_net_per_payer','Contribution per acquired payer over 24 months = 1.625 purchases x INR 43.32 x (1 + 40% year-two return) = INR 98.56. Scenario, not observed.')

,channel,label,budget,share_of_envelope,attributed_cac,cac_basis,cac_build,attributed_payers,incremental_cac_at_100,incremental_cac_at_87_5,incremental_cac_at_60,clears_200_at_87_5,net_per_payer_24m_at_100,net_per_payer_24m_at_87_5,channel_net_24m_at_100
0,owned_lifecycle,Owned lifecycle,9200000,0.303,99.256,Calculated,INR 0.8631 per WhatsApp message (O) / 1% conve...,"92,689.144",99.256,113.436,165.427,True,-0.696,-13.016,"-64,504.873"
1,native_personalities,Native personalities,4300000,0.141,150.000,Test ceiling,No public cost basis. Zero incremental rights ...,"28,666.667",150.000,171.429,250.000,True,-51.439,-63.759,"-1,474,596.906"
2,contests,Contests,2500000,0.082,158.974,Calculated,"(5,000 clicks x INR 10 CPC (O) + INR 12,000 fe...","15,725.806",158.974,181.685,264.957,True,-60.414,-72.734,"-950,055.504"
3,publisher_takeovers,Publisher takeovers,1200000,0.039,180.000,Test ceiling,No ESPN.in quote. Ceiling: fee <= INR 180 x ex...,"6,666.667",180.000,205.714,300.000,False,-81.439,-93.759,"-542,929.513"
4,paid_media_test,Paid media (capped test),2500000,0.082,333.333,Calculated,INR 10 CPC (O) / 3% click-to-purchase (A),"7,500.000",333.333,380.952,555.556,False,-234.773,-247.093,"-1,760,795.702"


,item,value
0,total,"30,400,000.000"
1,committed_acquisition,"19,700,000.000"
2,brand_and_measurement,"3,000,000.000"
3,performance_reserve,"7,700,000.000"
4,reserve_share,0.253
5,attributed_payers,"151,248.284"
6,attributed_blended_cac_incl_brand,150.084
7,attributed_blended_cac_media_only,130.249
8,incrementality_needed_for_200,0.750
9,owned_half_response_cac,198.513


,line,gate1,gate2,gate3,total,share_of_envelope
0,owned_lifecycle,"23,289.545","2,500,000.000","6,676,710.455",9200000,0.303
1,native_personalities,0.000,"1,000,000.000","3,300,000.000",4300000,0.141
2,contests,0.000,"600,000.000","1,900,000.000",2500000,0.082
3,publisher_takeovers,0.000,"300,000.000","900,000.000",1200000,0.039
4,paid_media_test,0.000,"2,500,000.000",0.000,2500000,0.082
5,brand_and_measurement,"76,710.455","600,000.000","2,323,289.545",3000000,0.099
6,performance_reserve,0.000,0.000,"7,700,000.000",7700000,0.253


<Figure size 1000x420 with 1 Axes>

## 3. Personas and capacity
In-app channels (owned, native) reach portfolio fans already on FanCode and are split in proportion to the base sport pools. External channels (contests, publisher, paid) reach the Slam Tourist. Pools can overlap and are a capacity check, not a forecast.

In [5]:
pp=pd.DataFrame([dict(persona=k,attributed_payers=v,share=res['personas']['payer_share'][k]) for k,v in res['personas']['payers'].items()]);display(table(pp,'08_persona_payers'))
pools=pd.DataFrame([dict(scenario=k,**{s+'_m':x for s,x in d.items()}) for k,d in res['personas']['pools_m'].items()]);pools['payer_target_share_of_pools']=[res['personas']['capacity_share_of_pools'][k] for k in pools.scenario];display(table(pools,'08_sport_pools_capacity'))

,persona,attributed_payers,share
0,Grid Strategist (F1),"40,389.132",0.267
1,Matchday Loyalist (football),"72,700.437",0.481
2,Throttle Rider (MotoGP),"8,266.242",0.055
3,Slam Tourist (external),"29,892.473",0.198


,scenario,f1_m,football_m,motogp_m,payer_target_share_of_pools
0,low,5.250,5.250,1.074,0.013
1,base,7.770,13.986,1.590,0.006
2,high,9.450,25.515,1.934,0.004


## 4. The upgrade cohort and the 24-month P&L
Eligible pass buyers = 24M ordinary-week viewers x 90% core (C, illustrative) x 20% reached (A) x 20% active ATP pass buyers (A) = 864k. The control upgrade rate is 10% (A). An incremental upgrade is worth the season contribution less the passes that buyer would have bought anyway, in year one and again at 85% renewal in year two. The credit is paid to every treated upgrader, including the 10% who would have upgraded anyway.

In [6]:
uc=pd.DataFrame([dict(item=k,value=v) for k,v in res['upgrade_cohort'].items()]);display(table(uc,'08_upgrade_cohort'))
grid=pd.DataFrame(res['pnl_grid']);display(table(grid,'08_pnl_grid'))
scen=pd.DataFrame([dict(scenario=k,**v) for k,v in res['scenarios'].items()]);display(table(scen,'08_pnl_scenarios'))
ref=res['reference_case']
steps=[('Acquired payers\nyear 1',ref['acquisition_y1']),('Upgrades, net\nof credit, year 1',ref['upgrade_y1']),('Committed spend\nincl. retention',-ref['spend']),('Year-one\nnet',None),('Year-two\ncontribution',ref['y2']),('Net at\n24 months',None)]
fig_,ax=plt.subplots(figsize=(10,4.6));run=0;tops=[0]
for i,(label,val) in enumerate(steps):
 if val is None:
  ax.bar(i,run/1e7,color='#193047',width=.6);ax.annotate(f'{run/1e7:+.2f}',(i,run/1e7),xytext=(0,4 if run>=0 else -12),textcoords='offset points',ha='center',fontsize=9);tops.append(run);continue
 ax.bar(i,val/1e7,bottom=run/1e7,color=COLORS[1] if val>=0 else COLORS[3],width=.6);end=run+val
 ax.annotate(f'{val/1e7:+.2f}',(i,max(run,end)/1e7),xytext=(0,4),textcoords='offset points',ha='center',fontsize=9);tops+= [run,end];run=end
ax.set_ylim(min(tops)/1e7-.35,max(tops)/1e7+.35)
ax.axhline(0,color='black',lw=.8);ax.set_xticks(range(len(steps)));ax.set_xticklabels([s[0] for s in steps],fontsize=8.5);ax.set_ylabel('INR crore')
ax.set_title(f"Reference case: 16% upgrade rate, 87.5% incrementality. Payback {ref['payback_months']:.1f} months",fontsize=12)
fig('08_reference_waterfall','Reference scenario only. It clears 24-month break-even but not the 15-month payback gate. Rights fee excluded.')

,item,value
0,core,"21,600,000.000"
1,reached,"4,320,000.000"
2,eligible_pass_buyers,"864,000.000"
3,control_rate,0.100
4,control_upgrades,"86,400.000"
5,value_per_incremental_upgrade_y1,228.319
6,value_per_incremental_upgrade_y2,194.071
7,retention_messaging_cost,"2,143,940.400"


,treatment_rate,incrementality,eligible_pass_buyers,incremental_upgrades,credit_cost,acquisition_y1,upgrade_y1,spend,y1_net,acquisition_y2,upgrade_y2,y2,net_24m,payback_months
0,0.120,1.000,"864,000.000","17,280.000","3,801,081.366","10,647,941.073","144,267.220","24,843,940.400","-14,051,732.107","4,259,176.429","3,353,546.298","7,612,722.727","-6,439,009.381",NaN
1,0.160,1.000,"864,000.000","51,840.000","5,068,108.488","10,647,941.073","6,767,937.270","24,843,940.400","-7,428,062.058","4,259,176.429","10,060,638.894","14,319,815.323","6,891,753.265",16.150
2,0.200,1.000,"864,000.000","86,400.000","6,335,135.609","10,647,941.073","13,391,607.319","24,843,940.400","-804,392.008","4,259,176.429","16,767,731.489","21,026,907.918","20,222,515.910",12.306
3,0.120,0.875,"864,000.000","17,280.000","3,801,081.366","9,316,948.438","144,267.220","24,843,940.400","-15,382,724.741","3,726,779.375","3,353,546.298","7,080,325.673","-8,302,399.068",NaN
4,0.160,0.875,"864,000.000","51,840.000","5,068,108.488","9,316,948.438","6,767,937.270","24,843,940.400","-8,759,054.692","3,726,779.375","10,060,638.894","13,787,418.269","5,028,363.577",17.082
5,0.200,0.875,"864,000.000","86,400.000","6,335,135.609","9,316,948.438","13,391,607.319","24,843,940.400","-2,135,384.642","3,726,779.375","16,767,731.489","20,494,510.865","18,359,126.223",12.834
6,0.120,0.600,"864,000.000","17,280.000","3,801,081.366","6,388,764.644","144,267.220","24,843,940.400","-18,310,908.536","2,555,505.857","3,353,546.298","5,909,052.155","-12,401,856.381",NaN
7,0.160,0.600,"864,000.000","51,840.000","5,068,108.488","6,388,764.644","6,767,937.270","24,843,940.400","-11,687,238.487","2,555,505.857","10,060,638.894","12,616,144.751","928,906.264",19.411
8,0.200,0.600,"864,000.000","86,400.000","6,335,135.609","6,388,764.644","13,391,607.319","24,843,940.400","-5,063,568.437","2,555,505.857","16,767,731.489","19,323,237.347","14,259,668.910",14.096


,scenario,treatment_rate,incrementality,eligible_pass_buyers,incremental_upgrades,credit_cost,acquisition_y1,upgrade_y1,spend,y1_net,acquisition_y2,upgrade_y2,y2,net_24m,payback_months
0,low,0.120,0.600,"864,000.000","17,280.000","3,801,081.366","6,388,764.644","144,267.220","24,843,940.400","-18,310,908.536","2,555,505.857","3,353,546.298","5,909,052.155","-12,401,856.381",NaN
1,reference,0.160,0.875,"864,000.000","51,840.000","5,068,108.488","9,316,948.438","6,767,937.270","24,843,940.400","-8,759,054.692","3,726,779.375","10,060,638.894","13,787,418.269","5,028,363.577",17.082
2,high,0.200,1.000,"864,000.000","86,400.000","6,335,135.609","10,647,941.073","13,391,607.319","24,843,940.400","-804,392.008","4,259,176.429","16,767,731.489","21,026,907.918","20,222,515.910",12.306


<Figure size 1000x460 with 1 Axes>

## 5. What the upgrade rate has to be
Two thresholds matter. **24-month break-even** is where the campaign repays its committed spend. **15-month payback** is the Gate 3 rule. Both are treatment upgrade rates against a 10% control.

In [7]:
be=[]
for q in m.v('acquired_payers','incrementality_scenarios'):
 be.append(dict(incrementality=q,net_without_upgrades=res['without_upgrades'][str(q)],break_even_rate=res['break_even_treatment'][str(q)],rate_for_15m_payback=res['treatment_for_15m_payback'][str(q)],break_even_if_reserve_deployed=res['break_even_treatment_reserve_deployed'][str(q)]))
be=pd.DataFrame(be);display(table(be,'08_break_even_rates'))
sens=pd.DataFrame([dict(active_pass_buyer_share=float(s),incrementality=float(q),break_even_rate=r) for s,d in res['break_even_treatment_by_pass_buyer_share'].items() for q,r in d.items()]);display(table(sens,'08_break_even_by_pass_buyer_share'))
ren=pd.DataFrame([dict(season_renewal=float(k),break_even_rate_at_87_5=v) for k,v in res['break_even_treatment_by_renewal'].items()]);display(table(ren,'08_break_even_by_renewal'))
fig_,ax=plt.subplots(figsize=(10,4.2));rates=np.linspace(.10,.24,57)
for q,c in zip([1.0,.875,.6],[COLORS[1],COLORS[0],COLORS[3]]):ax.plot(rates*100,[m.pnl(t,q)['net_24m']/1e7 for t in rates],color=c,label=f'{q:.1%} of acquired payers incremental')
ax.axhline(0,color='black',lw=.8);ax.axvline(10,color='grey',ls=':');ax.text(10.2,ax.get_ylim()[1]*.85,'control 10%',fontsize=8,color='grey')
ax.set_xlabel('Treatment upgrade rate among eligible pass buyers (%)');ax.set_ylabel('Net at 24 months, INR crore');ax.legend(fontsize=8)
b_,p_=res['break_even_treatment'],res['treatment_for_15m_payback']
ax.set_title(f"Break-even at {b_['1.0']:.1%}-{b_['0.6']:.1%}; 15-month payback only at {p_['1.0']:.1%}-{p_['0.6']:.1%}")
fig('08_break_even_curve','Base assumptions: 20% of reached core are active pass buyers, 85% renewal, INR 36.66 credit cost on every treated upgrader. Performance reserve unspent.')

,incrementality,net_without_upgrades,break_even_rate,rate_for_15m_payback,break_even_if_reserve_deployed
0,1.000,"-9,936,822.898",0.139,0.169,0.151
1,0.875,"-11,800,212.586",0.145,0.176,0.157
2,0.600,"-15,899,669.899",0.157,0.190,0.169


,active_pass_buyer_share,incrementality,break_even_rate
0,0.100,1.000,0.169
1,0.100,0.875,0.180
2,0.100,0.600,0.205
3,0.200,1.000,0.139
4,0.200,0.875,0.145
5,0.200,0.600,0.157
6,0.300,1.000,0.129
7,0.300,0.875,0.133
8,0.300,0.600,0.141


,season_renewal,break_even_rate_at_87_5
0,0.750,0.148
1,0.850,0.145
2,0.900,0.144


<Figure size 1000x420 with 1 Axes>

## 6. Sizing the gates to the economic test, not to mere detection
Gate 1 proves owned acquisition is incremental: the 95% lower bound on the lift must clear the lift at which incremental CAC equals INR 200. Gate 2 proves the upgrade engine: the 95% lower bound of (treatment − control) must clear the threshold minus the control rate. Both use 80% power. A true rate close to the threshold cannot be proved at any sensible cost; then the decision is continue or reallocate, never scale.

In [8]:
g1=pd.DataFrame([res['gate1_owned_pilot']]);display(table(g1,'08_gate1_owned_pilot'))
g2=pd.DataFrame(res['gate2_upgrade_test']);display(table(g2,'08_gate2_upgrade_test'))

,lift_threshold_pp,true_lift_pp,per_arm,total_users,treatment_messages_cost,treatment_cost_with_operations,implied_incremental_cac_at_true_lift
0,0.496,0.800,23464,46928,"20,251.778","23,289.545",124.071


,incrementality,threshold_rule,threshold_rate,true_rate,per_arm,total_users,message_cost,credit_face_value_paid,note
0,1.000,24m_break_even,0.139,0.180,"1,127.000","2,254.000","1,118.621","9,027.270",
1,1.000,24m_break_even,0.139,0.200,533.000,"1,066.000",529.037,"4,743.700",
2,1.000,24m_break_even,0.139,0.220,316.000,632.000,313.651,"3,093.640",
3,1.000,15m_payback,0.169,0.180,"15,436.000","30,872.000","15,321.233","123,642.360",
4,1.000,15m_payback,0.169,0.200,"2,043.000","4,086.000","2,027.810","18,182.700",
5,1.000,15m_payback,0.169,0.220,790.000,"1,580.000",784.126,"7,734.100",
6,0.875,24m_break_even,0.145,0.180,"1,515.000","3,030.000","1,503.736","12,135.150",
7,0.875,24m_break_even,0.145,0.200,647.000,"1,294.000",642.190,"5,758.300",
8,0.875,24m_break_even,0.145,0.220,365.000,730.000,362.286,"3,573.350",
9,0.875,15m_payback,0.176,0.180,"101,227.000","202,454.000","100,474.377","810,828.270",


## 7. Independent checks
Key outputs are recomputed with Decimal arithmetic outside the model code.

In [9]:
from decimal import Decimal as D
pass_c=D(89)/D('1.18')-D(89)*D('0.02')*D('1.18')-D(30)
season_c=D(399)/D('1.18')-D(399)*D('0.02')*D('1.18')-D(30)
credit=season_c-(D('354.5')/D('1.18')-D('354.5')*D('0.02')*D('1.18')-D(30))
owned=D('0.8631')/D('0.01')*D('1.15')
payers=D(9200000)/owned+D(4300000)/D(150)+D(2500000)/((D(50000)+D(12000))/D(390))+D(1200000)/D(180)+D(2500000)/(D(10)/D('0.03'))
spend=D(19700000)+D(3000000)+D(4320000)*D('0.8631')*D('0.5')*D('1.15')
acq24=payers*D('0.875')*D('1.625')*pass_c*D('1.4')
inc=D('0.06')*D(864000);upg=inc*(season_c-D('1.625')*pass_c)*(1+D('0.85'))-D('0.16')*D(864000)*credit
ref_net=acq24+upg-spend
checks={'pass_contribution':abs(float(pass_c)-res['unit_economics']['pass_contribution'])<1e-9,
 'season_contribution':abs(float(season_c)-res['unit_economics']['season_contribution'])<1e-9,
 'credit_cost':abs(float(credit)-res['unit_economics']['credit_cost_per_upgrade'])<1e-9,
 'attributed_payers':abs(float(payers)-res['envelope']['attributed_payers'])<1e-6,
 'reference_net_24m':abs(float(ref_net)-res['reference_case']['net_24m'])<1e-3,
 'break_even_is_zero':abs(m.pnl(res['break_even_treatment']['0.875'],0.875)['net_24m'])<1e-3,
 'payback_threshold_is_15':abs(m.pnl(res['treatment_for_15m_payback']['0.875'],0.875)['payback_months']-15)<1e-6,
 'envelope_reconciles':abs(sum(g['total'] for g in res['gate_allocation'])-30400000)<1e-6,
 'gates_reconcile':abs(gates.gate1.sum()-1e5)<1e-6 and abs(gates.gate2.sum()-75e5)<1e-6 and abs(gates.gate3.sum()-228e5)<1e-6,
 'only_owned_native_contests_clear_200_at_87_5':set(ch[ch.clears_200_at_87_5].channel)=={'owned_lifecycle','native_personalities','contests'},
 'upgrades_exclude_control':res['reference_case']['incremental_upgrades']==0.06*864000}
check('08_economics_model',checks)
r=res;ref=r['reference_case']
report('08_economics_findings',f"Committed acquisition of INR {r['envelope']['committed_acquisition']/1e7:.2f} Cr buys {r['envelope']['attributed_payers']/1e3:.1f}k attributed payers at INR {r['envelope']['attributed_blended_cac_incl_brand']:.0f} blended attributed CAC including brand and measurement, which meets INR 200 incremental CAC only at {r['envelope']['incrementality_needed_for_200']:.0%} incrementality or better. Each acquired payer contributes INR {r['unit_economics']['contribution_per_acquired_payer_24m']:.2f} over 24 months, so no channel repays its CAC on pass purchases alone; owned is closest. Without upgrades the campaign loses INR {-r['without_upgrades']['1.0']/1e7:.2f} to {-r['without_upgrades']['0.6']/1e7:.2f} Cr over 24 months. Against a 10% control upgrade rate among 864k eligible pass buyers, 24-month break-even needs a treatment upgrade rate of {r['break_even_treatment']['1.0']:.1%} to {r['break_even_treatment']['0.6']:.1%}, and 15-month payback needs {r['treatment_for_15m_payback']['1.0']:.1%} to {r['treatment_for_15m_payback']['0.6']:.1%}. The reference case (16%, 87.5%) nets INR {ref['net_24m']/1e7:.2f} Cr at 24 months with {ref['payback_months']:.1f}-month payback, so it would not pass Gate 3. Scale therefore waits for Gate 2 evidence. The performance reserve of INR {r['envelope']['performance_reserve']/1e7:.2f} Cr is released only on measured marginal incremental CAC at or below INR 200.")

,check,passed
0,pass_contribution,True
1,season_contribution,True
2,credit_cost,True
3,attributed_payers,True
4,reference_net_24m,True
5,break_even_is_zero,True
6,payback_threshold_is_15,True
7,envelope_reconciles,True
8,gates_reconcile,True
9,only_owned_native_contests_clear_200_at_87_5,True
